### Install mlflow libraries if you have not done it already

In [1]:
# !pip install mlflow snowflake-snowpark-python==1.0.0 pyarrow==8.0.0 scikit-learn==1.1.1

In [2]:
# !pip install snowflake_mlflow-0.0.1-py3-none-any.whl

### Import all libraries including snowpark related libraries

In [44]:
# Snowpark Imports
from snowflake.snowpark.session import Session
import snowflake.snowpark.functions as F
import snowflake.snowpark.types as T
from snowflake.snowpark.functions import sproc, udf, col, log, when, lit
# Other Imports
import pandas as pd
import mlflow
from mlflow.deployments import get_deploy_client
import json
import joblib
from pprint import pprint

# Import Snowflake Plugin for MLflow
from snowflake.ml.mlflow import create_session

## from DataBricks

import numpy as np
import pandas as pd
from datetime import date
from datetime import datetime
import os
import gc
import io

from scipy.stats.mstats import winsorize   
from sklearn import preprocessing

### Create the snowflake connection with cred.json connection details
### Create internal stage in the DB, Schema for storing Models and functions

In [2]:
# Reading Snowflake Connection Details
snowflake_connection_cfg = json.loads(open('cred.json').read())

# Creating Snowpark Session
mlflow_poc_session = Session.builder.configs(snowflake_connection_cfg).create()
mlflow_poc_session.sql("use role accountadmin").collect()
mlflow_poc_session.sql("use warehouse hmwh").collect()

# Create a fresh & new schema
# mlflow_poc_session.sql('CREATE OR REPLACE SCHEMA MLFLOW_POC_DEMO').collect()
# mlflow_poc_session.use_schema('MLFLOW_POC_DEMO')

# Creating stages for functions, models
# mlflow_poc_session.sql("CREATE STAGE IF NOT EXISTS FUNCTIONS").collect()
# mlflow_poc_session.sql("CREATE STAGE IF NOT EXISTS MODELS").collect()

[Row(status='Statement executed successfully.')]

### define the features (categorical and numeric) and Targets

In [3]:
VAR_PROFILE = []

VAR_TARGET = ['RETAIL_LTR', 'RETAIL_LTV', 'COM_LTR', 'COM_LTV']

FEATURES_CATEGORICAL = ['RTL_RECENCY_12MO', 'RTL_FREQUENCY_12MO', 'RTL_TENURE_12MO', 
                    'COM_RECENCY_12MO', 'COM_FREQUENCY_12MO', 'COM_TENURE_12MO',
                    'VERTICAL', 'EMP_SIZE',  'IS_TEACHER', 'IS_PARENT',]

FEATURES_NUMERIC = [
       'RETAIL_NET_SALES', 
       'RETAIL_NET_SALES_SUPPLIES', 
       'RETAIL_NET_SALES_COMPUTERS',
       'RETAIL_NET_SALES_FACILITIES', 'RETAIL_NET_SALES_FOOD',
       'RETAIL_NET_SALES_FURNITURES', 'RETAIL_NET_SALES_INK',
       'RETAIL_NET_SALES_MAIL', 'RETAIL_NET_SALES_SERVICES',
       'RETAIL_NET_SALES_PRINTERS', 'RETAIL_NET_SALES_PAPER',
       'RETAIL_NET_SALES_PRINT', 'RETAIL_NET_SALES_WARRANTIES',
       'RETAIL_NET_SALES_TECH_ACCESSORIES', 'RETAIL_NET_SALES_OTHER',
            
       'COM_NET_SALES',
       'COM_NET_SALES_SUPPLIES', 'COM_NET_SALES_COMPUTERS',
       'COM_NET_SALES_FACILITIES', 'COM_NET_SALES_FOOD',
       'COM_NET_SALES_FURNITURES', 'COM_NET_SALES_INK', 'COM_NET_SALES_MAIL',
       'COM_NET_SALES_SERVICES', 'COM_NET_SALES_PRINTERS',
       'COM_NET_SALES_PAPER', 'COM_NET_SALES_PRINT',
       'COM_NET_SALES_WARRANTIES', 'COM_NET_SALES_TECH_ACCESSORIES',
       'COM_NET_SALES_OTHER', 
    
       'RETAIL_MARGIN', 
       'RETAIL_MARGIN_SUPPLIES', 'RETAIL_MARGIN_COMPUTERS',
       'RETAIL_MARGIN_FACILITIES', 'RETAIL_MARGIN_FOOD',
       'RETAIL_MARGIN_FURNITURES', 'RETAIL_MARGIN_INK', 
       'RETAIL_MARGIN_MAIL',
       'RETAIL_MARGIN_SERVICES', 'RETAIL_MARGIN_PRINTERS',
       'RETAIL_MARGIN_PAPER', 'RETAIL_MARGIN_PRINT',
       'RETAIL_MARGIN_WARRANTIES', 'RETAIL_MARGIN_TECH_ACCESSORIES',
       'RETAIL_MARGIN_OTHER', 
    
       'COM_MARGIN', 'COM_MARGIN_SUPPLIES',
       'COM_MARGIN_COMPUTERS', 'COM_MARGIN_FACILITIES', 'COM_MARGIN_FOOD',
       'COM_MARGIN_FURNITURES', 'COM_MARGIN_INK', 'COM_MARGIN_MAIL',
       'COM_MARGIN_SERVICES', 'COM_MARGIN_PRINTERS', 'COM_MARGIN_PAPER',
       'COM_MARGIN_PRINT', 'COM_MARGIN_WARRANTIES',
       'COM_MARGIN_TECH_ACCESSORIES', 'COM_MARGIN_OTHER', 

        'RETAIL_SFC', 'RETAIL_VFC',
        'COM_SFC', 'COM_VFC',
        
       'RETAIL_NET_SALES_RETURNS', 'RETAIL_MARGIN_RETURNS',
       'COM_NET_SALES_RETURNS', 'COM_MARGIN_RETURNS',
    
       'RETAIL_NET_SALES_KIOSK', 'RETAIL_MARGIN_KIOSK', 
       'COM_NET_SALES_BOPIS', 'COM_MARGIN_BOPIS', 
       
       'RETAIL_SRWSTM', 'RETAIL_SRWINK', 
       'COM_SRWSTM', 'COM_SRWINK',
       'RETAIL_NET_SALES_STAPLES_BRAND', 'RETAIL_MARGIN_STAPLES_BRAND',
       'COM_NET_SALES_STAPLES_BRAND', 'COM_MARGIN_STAPLES_BRAND']

VAR_SUBSET = FEATURES_CATEGORICAL + FEATURES_NUMERIC + VAR_TARGET

VAR_TOTAL = VAR_SUBSET + VAR_PROFILE


In [36]:
def build_xg_model(p_df: pd.DataFrame,features:list,categorical:list):
    from sklearn.pipeline import Pipeline
    from sklearn.impute import SimpleImputer
    from sklearn.preprocessing import StandardScaler, OneHotEncoder
    from sklearn.compose import ColumnTransformer
    import xgboost as xgb
    
    numeric_features = features
    categorical_features = categorical

    feature_names = numeric_features + categorical_features

    numeric_transformer = Pipeline(steps=[
        # ('imputer', SimpleImputer(strategy='mean')),
        ('scaler', StandardScaler(with_mean=True,with_std=True))])

    categorical_transformer = Pipeline(steps=[
        # ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
        ('onehot', OneHotEncoder(handle_unknown='ignore'))])

    preprocessor = ColumnTransformer(
        transformers=[
            ('num', numeric_transformer, numeric_features),
            ('cat', categorical_transformer, categorical_features)])

    model = Pipeline(steps=[
                ('preprocessor', preprocessor),
                ('regressor'
                    ,xgb.XGBRegressor(seed = 20, 
                                objective='reg:squarederror', 
                                max_depth=6, 
                                n_estimators=140, 
                                learning_rate=0.1))
            ])

    return model


In [45]:
def save_file(session, model, path, dest_filename):
    import io
    # logger.debug('#save_file: -- START--')
    input_stream = io.BytesIO()
    joblib.dump(model, input_stream)
    session._conn.upload_stream(input_stream, path, dest_filename)
    return "successfully created file: " + path

In [49]:
@sproc(name='func_winsorize_std_scale_pipeline',
       packages=['snowflake-snowpark-python',
                 'pandas','scipy',
                 'scikit-learn==1.1.1',
                 'joblib',
                 'mlflow',
                 'xgboost==1.3.3'],
       stage_location='@MODELS',
       is_permanent=True,
       replace=True)
# def staples_train_rf_model(session: Session, training_table: str, sample_size_n: int, model_name: str,features:list, Y: str,test_size:float,random_state:int,ne:int,nj:int,cw:str, md:int) -> str:
def func_winsorize_std_scale_pipeline(session: Session, 
                             inp_table_name: str,
                             mincount: int,
                             features:list,
                             targets:list,
                             categorical:list,
                             stage_name: str,
                             model_name: str)-> str :
    from sklearn.metrics import accuracy_score, classification_report, precision_score, recall_score, confusion_matrix, RocCurveDisplay
    from sklearn import metrics
    # import matplotlib.pyplot as plt
#    training_data = session.table(training_table).sample(n=sample_size_n)
    training_data = session.table(inp_table_name).limit(mincount)
    test_size = 0.1
    Data_train, Data_test = training_data.random_split([1-test_size, test_size], seed=43)
    pd_Data_train=Data_train.to_pandas()
    pd_Data_test=Data_test.to_pandas()
    import xgboost 
    # Model building
    xg = build_xg_model(pd_Data_train[features+categorical],features,categorical)
    xg.fit(pd_Data_train[features+categorical], pd_Data_train[targets[1]])

    model_dir = '@MODELS'
    model_fl = model_name+'.joblib'
    save_file(session, xg, model_dir ,model_fl)

    score = xg.score(pd_Data_test[features+categorical], pd_Data_test[targets[1]])
    
    y_pred = xg.predict(pd_Data_test)
    # df_classification_report = get_classification_report(y_pred,pd_Data_test[Y]).reset_index().rename(columns={"index": "class"}).reset_index(drop=True)
    # df_model_info = get_model_info(model_fl,test_size,random_state,ne,nj,cw,md)
    # df_model_info=df_model_info.append([df_model_info]*5,ignore_index=True)
    # session.create_dataframe(df_classification_report.join(df_model_info)).write.mode("append").save_as_table("staples_model_output")
    
#     ax = plt.gca()
#     rfc_disp = RocCurveDisplay.from_estimator(rf, pd_Data_test[features], pd_Data_test[Y], ax=ax, alpha=0.8)
#     rfc_disp.plot(ax=ax, alpha=0.8)
    
#     return metrics.plot_roc_curve(rf, pd_Data_test[features], pd_Data_test[Y])
    return score

The version of package joblib in the local environment is 1.2.0, which does not fit the criteria for the requirement joblib. Your UDF might not work when the package version is different between the server and your local environment
The version of package mlflow in the local environment is 2.1.1, which does not fit the criteria for the requirement mlflow. Your UDF might not work when the package version is different between the server and your local environment
package xgboost is not installed in the local environmentYour UDF might not work when the package is installed on the server but not on your local environment.


In [50]:
func_winsorize_std_scale_pipeline('F_CLV_FEATURES_INPUT', 
                                                  19000000,
                                                  FEATURES_NUMERIC, 
                                                  VAR_TARGET,
                                                  FEATURES_CATEGORICAL,
                                                  'MODELS',
                                                  'LTV_STAPLES')

'0.03696729583660319'